In [1]:
import subprocess
import sys

def instalar_si_falta(paquete):
    try:
        __import__(paquete.replace("-", "_"))
        print(f"{paquete} ya está instalado")
    except ImportError:
        print(f"Instalando {paquete}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", paquete])

# Lista de paquetes a verificar
paquetes = ["langchain", "langchain_groq", "langchain_community", "ddgs", "dotenv"]

for pkg in paquetes:
    instalar_si_falta(pkg)

print("Todos los paquetes están listos!")

langchain ya está instalado
langchain_groq ya está instalado
langchain_community ya está instalado
ddgs ya está instalado
dotenv ya está instalado
Todos los paquetes están listos!


In [2]:
import os
from dotenv import load_dotenv
from langchain_community.tools import DuckDuckGoSearchResults

load_dotenv()

# Verificar que la API key existe
groq_api_key = os.getenv("APY_KEY_GROQ")
if not groq_api_key:
    raise ValueError("❌ GROQ_API_KEY no encontrada en .env")
else:
    print("✅ API Key de Groq cargada correctamente")

# Ver los primeros caracteres de la key (seguro)

✅ API Key de Groq cargada correctamente


In [3]:
from langchain_groq import ChatGroq

# Elige el modelo que prefieras
MODELO = "llama-3.3-70b-versatile"  # Potente y gratuito

llm = ChatGroq(
    groq_api_key=groq_api_key,
    model=MODELO,
    temperature=0.2,           # Baja temperatura = más consistente
    max_retries=2,             # Reintentar si falla
    timeout=60,                # Timeout en segundos
    verbose=True               # Mostrar logs detallados
)

print(f"✅ LLM de Groq inicializado")
print(f"   Modelo: {MODELO}")
print(f"   Temperatura: 0.2")

✅ LLM de Groq inicializado
   Modelo: llama-3.3-70b-versatile
   Temperatura: 0.2


In [4]:
# Celda 4: Prueba simple de Groq
from langchain_core.messages import HumanMessage, SystemMessage

mensajes = [
    SystemMessage(content="Eres un asistente útil. Responde de manera concisa."),
    HumanMessage(content="¿Cuál es la capital de Argentina?")
]

respuesta = llm.invoke(mensajes)
print("🤖 Respuesta de Groq:")
print("-" * 40)
print(respuesta.content)
print("-" * 40)
print(f"\n✅ Tokens usados: {respuesta.response_metadata.get('token_usage', {})}")

🤖 Respuesta de Groq:
----------------------------------------
La capital de Argentina es Buenos Aires.
----------------------------------------

✅ Tokens usados: {'completion_tokens': 9, 'prompt_tokens': 60, 'total_tokens': 69, 'completion_time': 0.009622615, 'completion_tokens_details': None, 'prompt_time': 0.005424871, 'prompt_tokens_details': None, 'queue_time': 0.152499097, 'total_time': 0.015047486}


In [14]:
# Celda: Clase Agent_Search corregida
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.tools import DuckDuckGoSearchResults
from typing import List, Dict, Any

class Agent_Search:
    def __init__(self, llm):
        self.llm = llm
        self.searching = DuckDuckGoSearchResults(
            output_format="list",
            num_results=3,  # Nota: es 'num_results', no 'max_results'
            backend="text"
        )
        print("✅ Agente de búsqueda inicializado")
        
        # Manejar diferentes formas de obtener el nombre del modelo
        modelo = getattr(llm, 'model_name', getattr(llm, 'model', 'Desconocido'))
        temperatura = getattr(llm, 'temperature', 'N/A')
        print(f"🧠 LLM: modelo {modelo}, temperatura {temperatura}")
        
    def buscar(self, query: str) -> Dict[str, Any]:
        """
        Realiza una búsqueda y devuelve resultados formateados.
        """
        print(f"\n🔍 BUSCANDO: '{query}'")
        
        try:
            # CORRECCIÓN 1: Usar invoke() en lugar de run()
            output = self.searching.invoke(query)
            
            # CORRECCIÓN 2: Verificar el tipo de output
            if isinstance(output, str):
                # Si es string, no podemos formatearlo como lista
                return {
                    "exito": True,
                    "Query": query,
                    "cantidad": 1,
                    "resultados": [{"contenido": output, "fuente": "DuckDuckGo"}],
                    "texto_completo": output,
                    "raw_output": output
                }
            
            # Si es lista, procesar normalmente
            output_formatted = self._formatear_output(output)
            
            return {
                "exito": True,
                "Query": query,
                "cantidad": len(output_formatted),
                "resultados": output_formatted,
                "texto_completo": self._a_texto(output_formatted),
                # CORRECCIÓN 3: DuckDuckGo NO tiene metadata de tokens
                # Solo el LLM tiene tokens, no la búsqueda
                "Tokens_usados": {}  # DuckDuckGo no tiene esta info
            }
            
        except Exception as e:
            print(f"❌ Error en búsqueda: {e}")
            return {
                "exito": False,
                "Query": query,
                "error": str(e),
                "resultados": []
            }
    
    def _formatear_output(self, resultados_raw: list) -> list:
        """Formatea los resultados de DuckDuckGo"""
        resultados = []
        
        for r in resultados_raw:
            resultado = {
                "titulo": r.get('title', 'Sin título'),
                "url": r.get('link', r.get('url', 'Sin URL')),
                "contenido": r.get('snippet', r.get('content', 'Sin contenido')),
                "fuente": "DuckDuckGo"
            }
            resultados.append(resultado)
            print(f"  📄 {resultado['titulo'][:60]}...")
            
        return resultados
    
    def _a_texto(self, resultados: list) -> str:
        """Convierte resultados a texto plano para usar en prompts"""
        texto = "RESULTADOS DE BÚSQUEDA:\n\n"
        
        for i, r in enumerate(resultados, 1):
            texto += f"--- FUENTE {i} ---\n"
            texto += f"Título: {r['titulo']}\n"
            texto += f"URL: {r['url']}\n"
            texto += f"Contenido:\n{r['contenido']}\n\n"
        
        return texto
    
    def resumir_query(self, query: str, resultado: Dict) -> str:
        """
        Resume los resultados usando el LLM.
        CORRECCIÓN 4: Usar invoke() en lugar de run()
        """
        if not resultado.get("exito", False):
            return f"No se pudo obtener resultados para resumir. Error: {resultado.get('error', 'Desconocido')}"
        
        prompt = ChatPromptTemplate.from_template("""Eres un asistente de investigación. Basado en la siguiente información de búsqueda,
        responde la pregunta del usuario de manera clara y concisa.
        
        PREGUNTA: {query}
        
        INFORMACIÓN ENCONTRADA:
        {informacion}
        
        INSTRUCCIONES:
        1. Responde directamente a la pregunta usando SOLO la información proporcionada
        2. Si la información es insuficiente, indícalo claramente
        3. Cita las fuentes cuando sea relevante
        4. Sé conciso (máximo 3-4 párrafos)
        
        RESPUESTA:
        """)
        
        cadena = prompt | self.llm
        # CORRECCIÓN 5: Usar invoke() con diccionario, no run()
        respuesta = cadena.invoke({"query": query, "informacion": resultado["texto_completo"]})
        
        return respuesta.content
    
    def resumir_query_con_tokens(self, query: str, resultado: Dict) -> Dict:
        """
        Versión que también devuelve información de tokens del LLM.
        """
        if not resultado.get("exito", False):
            return {
                "contenido": f"Error: {resultado.get('error', 'Desconocido')}",
                "tokens": {},
                "modelo": None
            }
        
        prompt = ChatPromptTemplate.from_template("""Eres un asistente de investigación. Basado en la siguiente información de búsqueda,
        responde la pregunta del usuario de manera clara y concisa.
        
        PREGUNTA: {query}
        
        INFORMACIÓN ENCONTRADA:
        {informacion}
        
        INSTRUCCIONES:
        1. Responde directamente a la pregunta usando SOLO la información proporcionada
        2. Si la información es insuficiente, indícalo claramente
        3. Cita las fuentes cuando sea relevante
        4. Sé conciso (máximo 3-4 párrafos)
        
        RESPUESTA:
        """)
        
        cadena = prompt | self.llm
        respuesta = cadena.invoke({"query": query, "informacion": resultado["texto_completo"]})
        
        # Extraer metadata de tokens
        metadata = respuesta.response_metadata
        token_usage = metadata.get('token_usage', {})
        
        return {
            "contenido": respuesta.content,
            "tokens": {
                "prompt_tokens": token_usage.get('prompt_tokens', 'N/A'),
                "completion_tokens": token_usage.get('completion_tokens', 'N/A'),
                "total_tokens": token_usage.get('total_tokens', 'N/A')
            },
            "modelo": metadata.get('model_name', 'Desconocido'),
            "respuesta_raw": respuesta  # Para depuración si es necesario
        }

In [24]:
# Celda 5: Probar búsqueda simple
agente=Agent_Search(llm)

query = input(("🔎 Ingresa tu consulta de búsqueda: "))
resultado = agente.buscar(query)

print(f"\n📊 RESULTADO:")
print(f"  ✅ Éxito: {resultado['exito']}")
print(f"  📝 Query: {resultado['Query']}")
print(f"  🔢 Resultados encontrados: {resultado['cantidad']}")
resumen = agente.resumir_query_con_tokens(query, resultado)
print(f"  📄 Resumen: {resumen['contenido']}")
if resumen['tokens']:
        print(f"  📥 Prompt tokens:     {resumen['tokens']['prompt_tokens']}")
        print(f"  📤 Completion tokens: {resumen['tokens']['completion_tokens']}")
        print(f"  🔢 Total tokens:      {resumen['tokens']['total_tokens']}")
        print(f"  🧠 Modelo:            {resumen['modelo']}")

✅ Agente de búsqueda inicializado
🧠 LLM: modelo llama-3.3-70b-versatile, temperatura 0.2

🔍 BUSCANDO: 'La Capital de Argentina es la pampa?'
  📄 Provincia de La Pampa - Wikipedia, la enciclopedia libre...
  📄 Región pampeana - Wikipedia, la enciclopedia libre...
  📄 Santa Rosa (La Pampa) - Wikipedia, la enciclopedia libre...

📊 RESULTADO:
  ✅ Éxito: True
  📝 Query: La Capital de Argentina es la pampa?
  🔢 Resultados encontrados: 3
  📄 Resumen: La pregunta es si la capital de Argentina es la Pampa. Según la información proporcionada, la respuesta es no. La Pampa es una provincia de Argentina, y su capital es Santa Rosa, como se menciona en la FUENTE 1 y la FUENTE 3. La región pampeana, que incluye la ciudad de Buenos Aires y varias provincias, es una región geográfica y cultural importante en Argentina, pero no es la capital del país, como se indica en la FUENTE 2.

No se proporciona información explícita sobre la capital de Argentina en las fuentes proporcionadas. Sin embargo, se puede